# Thermal Homography Quick Start

This notebook provides a quick introduction to the thermal homography project.

In [ ]:
import sys
sys.path.insert(0, '..')

import torch
import numpy as np
import matplotlib.pyplot as plt

print(f"PyTorch version: {torch.__version__}")
print(f"Device: {'cuda' if torch.cuda.is_available() else 'cpu'}")

## 1. Generate Synthetic Data

In [ ]:
from src.data.synthetic_generator import generate_checkerboard_pair, SyntheticThermalGenerator

# Generate a single pair
result = generate_checkerboard_pair()

fig, axes = plt.subplots(1, 3, figsize=(12, 4))

axes[0].imshow(result['image_src'], cmap='hot')
axes[0].set_title('Source Image')
axes[0].axis('off')

axes[1].imshow(result['image_tgt'], cmap='hot')
axes[1].set_title('Target Image (warped)')
axes[1].axis('off')

# Show homography
axes[2].text(0.1, 0.5, f"Homography:\n{result['homography']}", fontsize=10, family='monospace')
axes[2].axis('off')

plt.tight_layout()
plt.show()

## 2. Create and Test Model

In [ ]:
from src.models import ThermalHomographyNet

# Create model
model = ThermalHomographyNet(
    feature_dim=32,
    grid_size=16,  # 16x16 = 256 nodes (small for demo)
    gnn_num_layers=2,
    use_lrft=True,
    lrft_out_nodes=64,
)

# Count parameters
num_params = sum(p.numel() for p in model.parameters())
print(f"Model parameters: {num_params:,}")

In [ ]:
# Test forward pass
model.eval()

# Create batch from synthetic data
dataset = SyntheticThermalGenerator(n_samples=4, seed=42)

batch = [dataset[i] for i in range(4)]
image_src = torch.stack([b['image_src'] for b in batch])
image_tgt = torch.stack([b['image_tgt'] for b in batch])

print(f"Input shape: {image_src.shape}")

with torch.no_grad():
    output = model(image_src, image_tgt)

print(f"Homography output shape: {output['homography'].shape}")
print(f"Similarity matrix shape: {output['similarity'].shape}")

## 3. Test Loss Functions

In [ ]:
from src.training.losses import HomographyLoss, corner_loss

loss_fn = HomographyLoss()

H_pred = output['homography']
H_gt = torch.stack([b['homography_vec'] for b in batch])

losses = loss_fn(H_pred, H_gt)

print("Losses:")
for name, value in losses.items():
    print(f"  {name}: {value.item():.4f}")

## 4. Test E(2) Equivariance

In [ ]:
from src.models.e2_layers import E2EquivariantGNN
from src.utils.equivariance_tests import test_equivariance_numerical
from torch_geometric.nn import knn_graph

# Create E2 GNN
gnn = E2EquivariantGNN(
    in_channels=32,
    hidden_channels=64,
    out_channels=32,
    num_layers=2,
)
gnn.eval()

# Test data
x = torch.randn(100, 32)
pos = torch.randn(100, 2)
edge_index = knn_graph(pos, k=8)

# Test equivariance
results = test_equivariance_numerical(gnn, x, pos, edge_index)

print("E(2) Equivariance Test:")
print(f"  Angles tested: {results['angles']}")
print(f"  Errors: {[f'{e:.6f}' for e in results['errors']]}")
print(f"  Max error: {results['max_error']:.6f}")
print(f"  All passed: {results['all_passed']}")

## 5. Visualize Similarity Matrix

In [ ]:
# Get similarity matrix from model output
similarity = output['similarity'][0].detach().numpy()

plt.figure(figsize=(8, 8))
plt.imshow(similarity, cmap='viridis')
plt.colorbar(label='Similarity')
plt.title('Feature Similarity Matrix')
plt.xlabel('Target Nodes')
plt.ylabel('Source Nodes')
plt.show()

## Next Steps

1. Run Phase 1 validation: `python scripts/run_validation.py`
2. Download real thermal data: `python scripts/download_dataset.py --dataset roadscene`
3. Start full training with config files in `configs/`